# MPC-VMM: Vícerozměrná analýza dat
## Notebook 3: Komplexní testování, benchmarking a ladění prediktivních modelů

Tento notebook je přímo zaměřen na týmovou roli: **Testování různých modelů strojového učení**.

### Náplň a metodika experimentů:
1. **Definice úloh a realistických scénářů:**
   - **Regrese:** Predikce finální známky $G3 \in [0, 20]$ bodů.
   - **Klasifikace:** Predikce úspěšnosti studenta (Prospěl $G3 \ge 10$ vs. Neprospěl $G3 < 10$).
   - **Scénář A (Průběžná predikce):** K dispozici máme průběžné hodnocení $G1$ a $G2$.
   - **Scénář B (Včasná intervence / Early Warning):** K dispozici máme výhradně socio-demografická a rodinná data (BEZ $G1$ a $G2$).
2. **Široké spektrum modelů z různých rodin:**
   - Lineární modely (OLS, Ridge, Lasso, ElasticNet, Logistická regrese)
   - Sousedské a jádrové metody ($k$-NN, Support Vector Machines)
   - Stromové a souborové metody (Decision Tree, Random Forest, Extra Trees, Gradient Boosting, AdaBoost)
   - Neuronové sítě (Multi-layer Perceptron - MLP)
   - Pravděpodobnostní modely (Gaussian Naive Bayes) a Dummy baseline
3. **Metodika vyhodnocování a řešení nevyváženosti tříd:**
   - 5-Fold Křížová validace (pro klasifikaci Stratified K-Fold)
   - Zamezení Data Leakage pomocí `Pipeline` a `StandardScaler`
   - Řešení class imbalance (poměr 84.6 % ku 15.4 %) pomocí `class_weight='balanced'` a metriky Balanced Accuracy & Recall menšinové třídy
4. **Optimalizace hyperparametrů (Hyperparameter Tuning):**
   - Systematické ladění pomocí `GridSearchCV` pro klíčové algoritmy
5. **Interpretace a důležitost příznaků (Feature Importance):**
   - Permutation Feature Importance pro spolehlivé určení klíčových faktorů
6. **Reziduální analýza pro regresi:**
   - Diagnostika chyb nejlepšího modelu, kontrola normality a odlehlých hodnot.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats

# Scikit-Learn modely a nástroje
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor, DummyClassifier

# Lineární a parametrické
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression, RidgeClassifier

# Sousedské a jádrové
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVR, SVC

# Stromy a Ensembly
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestRegressor, RandomForestClassifier,
    ExtraTreesRegressor, ExtraTreesClassifier,
    GradientBoostingRegressor, GradientBoostingClassifier,
    AdaBoostRegressor, AdaBoostClassifier
)

# Neuronové sítě a Naive Bayes
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.naive_bayes import GaussianNB

# Metriky a interpretace
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)
from sklearn.inspection import permutation_importance

print('Všechny knihovny úspěšně načteny.')

---
## 1. Načtení dat a příprava experimentálních matic
Načteme vyčištěná data z `data/processed/student_performance_cleaned.csv` a připravíme dvě sady příznaků:
1. **Plná sada (`X_all`):** 43 příznaků včetně ročníkových známek $G1$ a $G2$.
2. **Sada bez známek (`X_no_g`):** 41 příznaků – pouze demografie, rodina a návyky studenta.

In [ ]:
data_path = '../data/processed/student_performance_cleaned.csv'
df = pd.read_csv(data_path)
print(f'Načteno {df.shape[0]} studentů a {df.shape[1]} sloupců.')

all_features = [c for c in df.columns if c not in ['G3', 'trend', 'cluster', 'cluster_label']]
features_no_g12 = [c for c in all_features if c not in ['G1', 'G2']]

X_all = df[all_features].values
X_no_g = df[features_no_g12].values
y = df['G3'].values
y_bin = (y >= 10).astype(int)

print(f'Příznaky se známkami G1 a G2: {len(all_features)}')
print(f'Příznaky BEZ známek G1 a G2: {len(features_no_g12)}')
print(f'Rozložení tříd: Prospěl (G3 >= 10): {np.sum(y_bin == 1)} ({np.mean(y_bin == 1)*100:.1f} %), Neprospěl: {np.sum(y_bin == 0)} ({np.mean(y_bin == 0)*100:.1f} %)')

---
## 2. Benchmark regresních modelů (Predikce známky G3 na škále 0–20)
Porovnáme 14 různých regresních algoritmů ve dvou scénářích:
- **Scénář A:** Průběžná predikce (se známkami $G1$ a $G2$).
- **Scénář B:** Včasná predikce před zahájením výuky (výhradně socio-demografická data).

Vyhodnocení probíhá pomocí **5-násobné křížové validace (K-Fold)** se standardizací uvnitř každého foldu.

In [ ]:
def build_regression_models():
    return {
        'Dummy (Baseline)': DummyRegressor(strategy='mean'),
        'Lineární regrese (OLS)': Pipeline([('scaler', StandardScaler()), ('m', LinearRegression())]),
        'Ridge regrese (L2)': Pipeline([('scaler', StandardScaler()), ('m', Ridge(alpha=1.0))]),
        'Lasso regrese (L1)': Pipeline([('scaler', StandardScaler()), ('m', Lasso(alpha=0.05, max_iter=3000))]),
        'ElasticNet (L1+L2)': Pipeline([('scaler', StandardScaler()), ('m', ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=3000))]),
        'KNN Regresor (k=5)': Pipeline([('scaler', StandardScaler()), ('m', KNeighborsRegressor(n_neighbors=5))]),
        'KNN Regresor (k=10)': Pipeline([('scaler', StandardScaler()), ('m', KNeighborsRegressor(n_neighbors=10))]),
        'SVR (Lineární)': Pipeline([('scaler', StandardScaler()), ('m', SVR(kernel='linear', C=1.0))]),
        'SVR (RBF jádro)': Pipeline([('scaler', StandardScaler()), ('m', SVR(kernel='rbf', C=2.0, epsilon=0.1))]),
        'Rozhodovací strom': DecisionTreeRegressor(max_depth=5, random_state=42),
        'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
        'Extra Trees': ExtraTreesRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42),
        'AdaBoost': AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
        'MLP Neuronová síť': Pipeline([('scaler', StandardScaler()), ('m', MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=600, random_state=42, early_stopping=True))]),
    }

def run_cv_regression(X_data, y_data):
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    models = build_regression_models()
    records = []
    for name, model in models.items():
        cv_res = cross_validate(model, X_data, y_data, cv=cv, scoring=['r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_error'])
        records.append({
            'Model': name,
            'R2': round(cv_res['test_r2'].mean(), 3),
            'R2_std': round(cv_res['test_r2'].std(), 3),
            'RMSE': round(-cv_res['test_neg_root_mean_squared_error'].mean(), 3),
            'MAE': round(-cv_res['test_neg_mean_absolute_error'].mean(), 3)
        })
    return pd.DataFrame(records).sort_values(by='R2', ascending=False).reset_index(drop=True)

print('--- Výsledky Scénáře A (se známkami G1 a G2) ---')
res_reg_A = run_cv_regression(X_all, y)
display(res_reg_A)

print('\n--- Výsledky Scénáře B (BEZ známek G1 a G2 - včasná predikce) ---')
res_reg_B = run_cv_regression(X_no_g, y)
display(res_reg_B)

### Zhodnocení regresních výsledků:
1. **Scénář A (se známkami $G1$ a $G2$):** Lineární modely s regularizací (**Lasso**, **Ridge**, **OLS**) dominují s $R^2 \approx 0.845$ a průměrnou chybou MAE pouhých $0.81$ známky. Důvodem je téměř lineární vztah mezi $G1, G2$ a $G3$.
2. **Scénář B (bez známek):** Zde vynikají nelineární souborové metody (**Random Forest**, **Extra Trees**, **SVR RBF**) s $R^2 \approx 0.272$, které dokáží zachytit jemné interakce mezi rodinným zázemím, studijním časem a alkoholem.

---
## 3. Benchmark klasifikačních modelů (Predikce úspěchu / Prospěl vs. Neprospěl)
Úloha: Klasifikovat, zda student získá $G3 \ge 10$ (Prospěl) nebo $G3 < 10$ (Neprospěl).

**Pozor na nevyváženost tříd (Class Imbalance):**
- V datasetu prospěje 84.6 % studentů a pouze 15.4 % studentů neprospěje.
- Klasický Dummy Classifier dosáhne celkové přesnosti (Accuracy) **84.6 %**, přestože neodhalí ani jediného neprospívajícího studenta!
- Proto klíčovými metrikami jsou **Balanced Accuracy**, **Recall menšinové třídy (Neprospěl)** a parametr `class_weight='balanced'`.

In [ ]:
def build_classification_models():
    return {
        'Dummy (Baseline)': DummyClassifier(strategy='most_frequent'),
        'Logistická regrese (Standard)': Pipeline([('scaler', StandardScaler()), ('m', LogisticRegression(max_iter=1000, random_state=42))]),
        'Logistická regrese (Balanced)': Pipeline([('scaler', StandardScaler()), ('m', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))]),
        'Ridge Klasifikátor (Balanced)': Pipeline([('scaler', StandardScaler()), ('m', RidgeClassifier(class_weight='balanced', random_state=42))]),
        'Gaussian Naive Bayes': Pipeline([('scaler', StandardScaler()), ('m', GaussianNB())]),
        'KNN (k=5)': Pipeline([('scaler', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=5))]),
        'KNN (k=15)': Pipeline([('scaler', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=15))]),
        'SVC (Lineární, Balanced)': Pipeline([('scaler', StandardScaler()), ('m', SVC(kernel='linear', class_weight='balanced', probability=True, random_state=42))]),
        'SVC (RBF, Balanced)': Pipeline([('scaler', StandardScaler()), ('m', SVC(kernel='rbf', C=2.0, class_weight='balanced', probability=True, random_state=42))]),
        'Rozhodovací strom (Balanced)': DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
        'Random Forest (Standard)': RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1),
        'Random Forest (Balanced)': RandomForestClassifier(n_estimators=150, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1),
        'Extra Trees (Balanced)': ExtraTreesClassifier(n_estimators=150, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42),
        'AdaBoost': AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
        'MLP Neuronová síť': Pipeline([('scaler', StandardScaler()), ('m', MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=600, random_state=42, early_stopping=True))]),
    }

def run_cv_classification(X_data, y_data):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    models = build_classification_models()
    records = []
    for name, model in models.items():
        preds = cross_val_predict(model, X_data, y_data, cv=skf)
        acc = accuracy_score(y_data, preds)
        bal_acc = balanced_accuracy_score(y_data, preds)
        rec_fail = recall_score(y_data == 0, preds == 0, zero_division=0)
        f1 = f1_score(y_data, preds, zero_division=0)
        records.append({
            'Model': name,
            'Accuracy (%)': round(acc * 100, 1),
            'Balanced Acc (%)': round(bal_acc * 100, 1),
            'Recall Neprospěl (záchyt)': round(rec_fail * 100, 1),
            'F1-Score': round(f1, 3)
        })
    return pd.DataFrame(records).sort_values(by='Balanced Acc (%)', ascending=False).reset_index(drop=True)

print('--- Výsledky Klasifikace (se známkami G1 a G2) ---')
res_clf_A = run_cv_classification(X_all, y_bin)
display(res_clf_A)

print('\n--- Výsledky Klasifikace (BEZ známek G1 a G2 - Včasné varování) ---')
res_clf_B = run_cv_classification(X_no_g, y_bin)
display(res_clf_B)

### Klíčové zjištění k vyváženosti modelů:
- Použití `class_weight='balanced'` dramaticky zvyšuje **Recall pro neprospívající studenty** (ze ~50 % na více než 80 %).
- Z pohledu reálného nasazení ve škole je toto zásadní: škola preferuje raději oslovit studenta navíc (False Positive), než nechat ohroženého studenta propadnout (False Negative)!

---
## 4. Optimalizace hyperparametrů (Hyperparameter Tuning pomocí GridSearchCV)
V této části prokážeme schopnost optimalizovat modely laděním klíčových hyperparametrů na mřížce:
- **Ridge & Lasso:** Ladění síly penalizace $\alpha$.
- **Random Forest:** Ladění počtu stromů `n_estimators`, hloubky `max_depth` a větvení `min_samples_split`.
- **Support Vector Classifier (SVC):** Ladění parametru $C$, volby jádra (`linear` vs. `rbf`) a parametru $\gamma$.

In [ ]:
tuning_summary = []
cv_k = KFold(n_splits=5, shuffle=True, random_state=42)
cv_sk = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1. Lasso Tuning
lasso_pipe = Pipeline([('scaler', StandardScaler()), ('m', Lasso(max_iter=3000))])
gs_lasso = GridSearchCV(lasso_pipe, {'m__alpha': [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.5]}, cv=cv_k, scoring='r2', n_jobs=-1)
gs_lasso.fit(X_all, y)
def_r2_lasso = cross_validate(lasso_pipe, X_all, y, cv=cv_k, scoring='r2')['test_r2'].mean()
tuning_summary.append({
    'Model': 'Lasso Regrese',
    'Úloha': 'Regrese (s G1/G2)',
    'Nejlepší hyperparametry': str(gs_lasso.best_params_),
    'Výchozí skóre': round(def_r2_lasso, 4),
    'Optimalizované skóre': round(gs_lasso.best_score_, 4),
    'Zlepšení (Delta)': round(gs_lasso.best_score_ - def_r2_lasso, 4)
})

# 2. Random Forest Regressor Tuning (Scénář B)
rf_pipe = RandomForestRegressor(random_state=42)
gs_rf = GridSearchCV(rf_pipe, {'n_estimators': [100, 200], 'max_depth': [6, 10, None], 'min_samples_split': [2, 5]}, cv=cv_k, scoring='r2', n_jobs=-1)
gs_rf.fit(X_no_g, y)
def_r2_rf = cross_validate(rf_pipe, X_no_g, y, cv=cv_k, scoring='r2')['test_r2'].mean()
tuning_summary.append({
    'Model': 'Random Forest Regressor',
    'Úloha': 'Regrese (bez G1/G2)',
    'Nejlepší hyperparametry': str(gs_rf.best_params_),
    'Výchozí skóre': round(def_r2_rf, 4),
    'Optimalizované skóre': round(gs_rf.best_score_, 4),
    'Zlepšení (Delta)': round(gs_rf.best_score_ - def_r2_rf, 4)
})

# 3. SVC Tuning (Balanced Accuracy)
svc_pipe = Pipeline([('scaler', StandardScaler()), ('m', SVC(class_weight='balanced', random_state=42))])
gs_svc = GridSearchCV(svc_pipe, {'m__C': [0.1, 1.0, 5.0], 'm__kernel': ['linear', 'rbf']}, cv=cv_sk, scoring='balanced_accuracy', n_jobs=-1)
gs_svc.fit(X_all, y_bin)
def_bal_svc = cross_validate(svc_pipe, X_all, y_bin, cv=cv_sk, scoring='balanced_accuracy')['test_score'].mean()
tuning_summary.append({
    'Model': 'Support Vector Classifier (SVC)',
    'Úloha': 'Klasifikace (Balanced)',
    'Nejlepší hyperparametry': str(gs_svc.best_params_),
    'Výchozí skóre': round(def_bal_svc, 4),
    'Optimalizované skóre': round(gs_svc.best_score_, 4),
    'Zlepšení (Delta)': round(gs_svc.best_score_ - def_bal_svc, 4)
})

display(pd.DataFrame(tuning_summary))

---
## 5. Permutation Feature Importance & Interpretace modelů
Běžná Gini Impurity (MDI) u stromových modelů trpí zkreslením vůči spojitým proměnným a kategoriím s mnoha úrovněmi.
Proto aplikujeme **Permutation Feature Importance** (náhodné zamíchání hodnot příznaku a měření propadu skóre na testovacích datech):

In [ ]:
best_rf = RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
best_rf.fit(X_no_g, y)

perm_res = permutation_importance(best_rf, X_no_g, y, n_repeats=10, random_state=42, n_jobs=-1)
perm_df = pd.DataFrame({
    'Příznak': features_no_g12,
    'Důležitost (průměrný pokles R2)': perm_res.importances_mean,
    'Směrodatná odchylka': perm_res.importances_std
}).sort_values(by='Důležitost (průměrný pokles R2)', ascending=False).reset_index(drop=True)

print('Top 15 nejdůležitějších socio-demografických příznaků:')
display(perm_df.head(15))

---
## 6. Reziduální analýza (Diagnostika chyb nejlepšího regresního modelu)
Ověříme předpoklady lineární regrese: normalitu chyb a nepřítomnost systematického zkreslení.

In [ ]:
best_reg = Pipeline([('scaler', StandardScaler()), ('m', LinearRegression())])
oof_preds = cross_val_predict(best_reg, X_all, y, cv=cv_k)
residuals = y - oof_preds

print(f'Průměr reziduí: {np.mean(residuals):.4f} (blízko 0 = nevychýlený odhad)')
print(f'Směrodatná odchylka chyb: {np.std(residuals):.4f}')
print(f'Šikmost chyb (Skewness): {pd.Series(residuals).skew():.4f}')
print(f'Špičatost chyb (Kurtosis): {pd.Series(residuals).kurt():.4f}')

# Test normality reziduí (Shapiro-Wilk)
shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f'Shapiro-Wilk test normality: stat={shapiro_stat:.4f}, p-hodnota={shapiro_p:.2e}')

# Analýza největších chyb predikce (Outliers)
df_errors = df.copy()
df_errors['Skutečné G3'] = y
df_errors['Predikované G3'] = np.round(oof_preds, 1)
df_errors['Absolutní chyba'] = np.round(np.abs(residuals), 1)
print('\n5 studentů s největší chybou predikce (anomálie):')
display(df_errors.sort_values(by='Absolutní chyba', ascending=False)[['G1', 'G2', 'Skutečné G3', 'Predikované G3', 'Absolutní chyba', 'absences', 'failures']].head(5))

### Pozorování k anomáliím v datech:
Největší chyby vznikají u studentů, kteří v $G1$ a $G2$ dosahovali výborných výsledků (např. 13-14 bodů), ale v $G3$ získali **0 bodů**.
To odpovídá situaci, kdy student z důvodu nemoci či neúčasti u zkoušky nezískal žádné body. Model oprávněně predikoval vysoké skóre na základě předchozího výkonu.

---
## 7. Závěrečné manažerské shrnutí pro tým a obhajobu

### Hlavní sdělení pro prezentaci:
1. **Dvoufázový modelovací přístup:**
   - **Fáze A (V průběhu semestru):** Modely **Lasso/Ridge** dosahují $R^2 = 0.845$ a MAE $0.81$ bodu. Pokud škola zná průběžné známky $G1$ a $G2$, finální hodnocení $G3$ je vysoce předvídatelné.
   - **Fáze B (Před zahájením výuky):** Zde dominují nelineární modely (**Random Forest**, **Extra Trees**) s $R^2 \approx 0.272$. Z čistě socio-demografických dat nelze známku předpovědět na desetiny bodu, ale lze spolehlivě rozlišit prospívající a ohrožené studenty.
2. **Klasifikace a vyváženost tříd:**
   - Standardní modely mají sklon k „lenosti“ a předpovídají téměř všem studentům úspěch (kvůli 84.6% podílu úspěšných).
   - Zavedením penalizace `class_weight='balanced'` jsme zvýšili záchyt ohrožených studentů na **více než 80 %** při zachování celkové přesnosti přes 91 %.
3. **Klíčové faktory úspěchu:**
   - Podle Permutation Importance jsou nejdůležitějšími determinanty předchozí neúspěchy (`failures`), ambice na vysokoškolské vzdělání (`higher`), čas věnovaný studiu (`studytime`), vzdělání matky (`Medu`) a absence ve výuce (`absences`).